# Обработка категориальных столбцов и заполнение Nan

In [1]:
import pandas as pd  
import numpy as np  
import re
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, TargetEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [2]:
data = pd.read_csv('total_laggs.csv', low_memory=False)
data.head(3)

,student_id,program,education_level,academic_year,place_type,course,absence_status,discipline,module,exam_type,...,exam_type_prev,grade_prev,difficulty_prev,std_deviation_prev,students_responsed_ratio_prev,avg_grade_prev,min_prev,max_prev,target_grade,target_type
0,1,Прикладная математика и информатика,Бакалавриат,2024,Бюджетные,4,\N,Веб-поиск и ранжирование,1,Первая сдача,...,NaN,NaN,NaN,NaN,NaN,7.384615,4.0,10.0,7.0,Первая сдача
1,1,Прикладная математика и информатика,Бакалавриат,2024,Бюджетные,4,\N,Правовая грамотность,2,Первая сдача,...,NaN,NaN,NaN,NaN,NaN,7.647059,4.0,10.0,8.0,Первая сдача
2,2,Прикладной анализ данных и искусственный интел...,Бакалавриат,2022,Коммерческие за счет средств вуза,4,\N,Дискретная математика,1,Первая сдача,...,NaN,NaN,NaN,NaN,NaN,7.000000,7.0,7.0,7.0,Первая сдача


In [3]:
data.columns

Index(['student_id', 'program', 'education_level', 'academic_year',
       'place_type', 'course', 'absence_status', 'discipline', 'module',
       'exam_type', 'grade_10', 'difficulty_avg_score', 'std_deviation',
       'students_responsed_ratio', 'exam_type_prev', 'grade_prev',
       'difficulty_prev', 'std_deviation_prev',
       'students_responsed_ratio_prev', 'avg_grade_prev', 'min_prev',
       'max_prev', 'target_grade', 'target_type'],
      dtype='str')

In [4]:
columns_to_fill = ['program', 'education_level', 'place_type', 'absence_status', 'exam_type', 
                   'difficulty_avg_score', 'std_deviation', 'students_responsed_ratio', 
                   'exam_type_prev', 'grade_prev', 'difficulty_prev', 'std_deviation_prev', 
                   'students_responsed_ratio_prev']

for c in columns_to_fill:
    k = len(data[c].unique())
    print(f'{c}:  {k} unique')

program:  10 unique
education_level:  2 unique
place_type:  8 unique
absence_status:  3 unique
exam_type:  4 unique
difficulty_avg_score:  66 unique
std_deviation:  75 unique
students_responsed_ratio:  52 unique
exam_type_prev:  5 unique
grade_prev:  12 unique
difficulty_prev:  42 unique
std_deviation_prev:  43 unique
students_responsed_ratio_prev:  35 unique


- `education_level`: LabelEncoder
    - всего 2 значения 
- `place_type`: порядок + LabelEncoder
    - 8 значений, которым можно задать порядок 
- `exam_type`: LabelEncoder
    - 4 значения, порядок был задан ранее
- `absence_status`: LabelEncoder
    - 3 значения
- `discipline`: TargetEncoder
    - 106 значений 
- `program`: TargetEncoder
    - 10 значений
- `student_responsed_ratio`: интерполяция 
- `_prev` _(лаги)_: группировка + среднее значение

In [5]:
def transform_data(path_to_data):
    data = pd.read_csv(path_to_data, low_memory=False)

    # изменяем education_level: 
    education_encoder = LabelEncoder()
    education_encoder.fit(['Бакалавриат', 'Магистратура'])
    data['education_level'] = education_encoder.transform(data['education_level'])

    # изменяем exam_type: 
    exam_encoder = LabelEncoder()
    exam_encoder.fit(['Первая сдача', 'Пересдача по уважительной причине', 'Пересдача','Пересдача с комиссией'])
    data['exam_type'] = exam_encoder.transform(data['exam_type'])
    data['target_type'] = exam_encoder.transform(data['target_type'])

    # изменяем absence_status:
    absence_encoder = LabelEncoder()
    absence_encoder.fit(['valid', 'invalid', '\\N'])
    data['absence_status'] = absence_encoder.transform(data['absence_status'])

    # изменяем discipline и program:
    disc_encoder = TargetEncoder()
    data['discipline'] = disc_encoder.fit_transform(data[['discipline']], data['target_grade'])

    program_encoder = TargetEncoder()
    data['program'] = program_encoder.fit_transform(data[['program']], data['target_grade'])

    # difficulty_avg_score
    # берем среднее значение оценки сложности по конкретной дисциплине
    data['difficulty_avg_score'] = (data.groupby('discipline')['difficulty_avg_score'].transform(lambda x: x.fillna(x.mean())))
    data['difficulty_avg_score'] = (data['difficulty_avg_score'].fillna(data['difficulty_avg_score'].mean()))

    # std_deviation
    # разброс оценок тоже группируем по дисциплине и берем среднее
    data['std_deviation'] = (data.groupby('discipline')['std_deviation'].transform(lambda x: x.fillna(x.mean())))
    data['std_deviation'] = (data['std_deviation'].fillna(data['std_deviation'].mean()))
    
    # лаги:
    # тип экзамена: группируем по студентам и заполняем самым частым типом для каждого студента отдельно, если 
    # пропуски остались, заполняем самым частым типом для дисциплины, а потом самым частым типом глобально для всех студентов 
    data['exam_type_prev'] = (data.groupby('student_id')['exam_type_prev'].transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan)))
    data['exam_type_prev'] = (data.groupby('discipline')['exam_type_prev'].transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan)))
    data['exam_type_prev'] = (data['exam_type_prev'].fillna(data['exam_type_prev'].mode()[0]))
    data['exam_type_prev'] = exam_encoder.transform(data['exam_type_prev'])

    # оценка: группируем по студентам и заполняем медианным значением для каждого студента отдельно, если 
    # пропуски остались, заполняем медианой по дисциплине, а потом уже глобальным медианным значением для всех студентов 
    data['grade_prev'] = (data.groupby('student_id')['grade_prev'].transform(lambda x: x.fillna(x.median())))
    data['grade_prev'] = (data.groupby('discipline')['grade_prev'].transform(lambda x: x.fillna(x.median())))
    data['grade_prev'] = (data['grade_prev'].fillna(data['grade_prev'].median()))

    # берем среднее значение оценки сложности по конкретной дисциплине
    data['difficulty_prev'] = (data.groupby('discipline')['difficulty_prev'].transform(lambda x: x.fillna(x.mean())))
    data['difficulty_prev'] = (data['difficulty_prev'].fillna(data['difficulty_prev'].mean()))

    # разброс оценок тоже группируем по дисциплине и берем среднее
    data['std_deviation_prev'] = (data.groupby('discipline')['std_deviation_prev'].transform(lambda x: x.fillna(x.mean())))
    data['std_deviation_prev'] = (data['std_deviation_prev'].fillna(data['std_deviation_prev'].mean()))

    # доля ответивших: сначала группируем программу + курс + учебный год
    data['students_responsed_ratio_prev'] = (data.groupby(['program', 'course', 'academic_year'])['students_responsed_ratio_prev'].transform(lambda x: x.fillna(x.mean())))
    # потом курс + учебный год
    data['students_responsed_ratio_prev'] = (data.groupby(['course', 'academic_year'])['students_responsed_ratio_prev'].transform(lambda x: x.fillna(x.mean())))
    # потом уже глобально 
    data['students_responsed_ratio_prev'] = (data['students_responsed_ratio_prev'].fillna(data['students_responsed_ratio_prev'].mean()))

    data['students_responsed_ratio'] = (data.groupby(['program', 'course', 'academic_year'])['students_responsed_ratio'].transform(lambda x: x.fillna(x.mean())))
    data['students_responsed_ratio'] = (data.groupby(['course', 'academic_year'])['students_responsed_ratio'].transform(lambda x: x.fillna(x.mean())))
    data['students_responsed_ratio'] = (data['students_responsed_ratio'].fillna(data['students_responsed_ratio'].mean()))

    place_order = [['Без вступительных испытаний', 'Внеконкурсное поступление',
        'Бюджетные', 'Целевые', 'По межправительственным соглашениям',
        'Коммерческие за счет средств вуза', 'Коммерческие', 'Коммерческие места для иностранных граждан']]

    place_encoder = OrdinalEncoder(categories=place_order)
    data['place_type'] = place_encoder.fit_transform(data[['place_type']])
    
    return data

In [6]:
data_new = transform_data('total_laggs.csv')
data_new.head()

,student_id,program,education_level,academic_year,place_type,course,absence_status,discipline,module,exam_type,...,exam_type_prev,grade_prev,difficulty_prev,std_deviation_prev,students_responsed_ratio_prev,avg_grade_prev,min_prev,max_prev,target_grade,target_type
0,1,0.075439,0,2024,0.007737,4,0,0.011610,1,0,...,0,0.150876,0.179850,0.161501,0.142149,0.053189,2.0,0.000000,0.0,0
1,1,0.075439,0,2024,0.007737,4,0,0.011610,2,0,...,0,0.150876,0.179850,0.161501,0.142149,0.053189,2.0,0.000000,0.0,0
2,2,0.104502,0,2022,0.008711,4,0,0.012199,1,0,...,0,0.156795,0.130676,0.152440,0.118473,0.040948,5.0,0.076667,0.0,0
3,2,0.098055,0,2022,0.008756,4,0,0.014889,2,0,...,0,0.163742,0.133988,0.151490,0.113845,0.040288,5.0,0.069534,0.0,0
4,2,0.098055,0,2022,0.008756,4,0,0.014889,2,0,...,0,0.163742,0.133988,0.151490,0.113845,0.040288,5.0,0.076667,0.0,0


In [7]:
data_new.shape

(3360, 24)

In [8]:
data_new = data_new.dropna()
data_new.shape

(3360, 24)

In [9]:
data_new.to_csv('encoded_data.csv', index=False)